In [1]:
import time
start_time = time.time()

In [2]:
%load_ext cudf.pandas

In [3]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [4]:
from pathlib import Path
# 
import numpy as np
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time

In [5]:
passmark = 40

In [6]:
### cell 0 ###

### cell 0 (optimized)
benchmark_name = "student-performance-in-exams"
# read into a cuDF-backed DataFrame
df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent
    / "input"
    / "StudentsPerformance.csv"
)
factor = 1000
# instead of concatenating a list of 1000 copies, use a single GPU take() with a repeated index
df = df.take(df.index.repeat(factor))
# df.take on a repeated index will automatically produce a new RangeIndex of the expanded length
df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 1000000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count    Dtype
---  ------                       --------------    -----
 0   gender                       1000000 non-null  object
 1   race/ethnicity               1000000 non-null  object
 2   parental level of education  1000000 non-null  object
 3   lunch                        1000000 non-null  object
 4   test preparation course      1000000 non-null  object
 5   math score                   1000000 non-null  int64
 6   reading score                1000000 non-null  int64
 7   writing score                1000000 non-null  int64
dtypes: int64(3), object(5)
memory usage: 89.5+ MB


In [7]:
### cell 1 ###

df.isna().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

In [8]:
### cell 2 ###

df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
0,female,group B,bachelor's degree,standard,none,72,72,74
0,female,group B,bachelor's degree,standard,none,72,72,74
0,female,group B,bachelor's degree,standard,none,72,72,74
0,female,group B,bachelor's degree,standard,none,72,72,74


In [9]:
### cell 3 ###

df.describe()

,math score,reading score,writing score
count,1000000.000000,1000000.000000,1000000.000000
mean,66.089000,69.169000,68.054000
std,15.155504,14.592897,15.188065
min,0.000000,17.000000,10.000000
25%,57.000000,59.000000,57.750000
50%,66.000000,70.000000,69.000000
75%,77.000000,79.000000,79.000000
max,100.000000,100.000000,100.000000


In [10]:
### cell 4 ###

df.isnull().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

In [11]:
### cell 5 ###

# Convert 'math score' to numeric on GPU
df['math score'] = pd.to_numeric(df['math score'], errors='coerce')

# Initialize all as 'P' then GPU‐native mask to set failing scores to 'F'
df['Math_PassStatus'] = 'P'
df['Math_PassStatus'] = df['Math_PassStatus'].mask(
    df['math score'] < passmark,
    'F'
)

# Count values (GPU)
df.Math_PassStatus.value_counts()

Math_PassStatus
P    960000
F     40000
Name: count, dtype: int64

In [12]:
### cell 6 ###

# Convert reading score to numeric (GPU)
df['reading score'] = pd.to_numeric(df['reading score'], errors='coerce')

# Initialize all as 'P' (on GPU)
df['Reading_PassStatus'] = 'P'

# Overwrite with 'F' where score is below passmark (GPU‐native boolean indexing)
df.loc[df['reading score'] < passmark, 'Reading_PassStatus'] = 'F'

# Count pass/fail (GPU)
df['Reading_PassStatus'].value_counts()

Reading_PassStatus
P    974000
F     26000
Name: count, dtype: int64

In [13]:
### cell 7 ###

# Convert writing score to numeric on GPU
df['writing score'] = pd.to_numeric(df['writing score'], errors='coerce')

# Build a boolean mask for failures (writing score < passmark)
mask = df['writing score'] < passmark

# Initialize all as pass ('P') and then GPU‐accelerated mask failures to 'F'
df['Writing_PassStatus'] = 'P'
df['Writing_PassStatus'] = df['Writing_PassStatus'].mask(mask, 'F')

# Compute counts on GPU
df['Writing_PassStatus'].value_counts()

Writing_PassStatus
P    968000
F     32000
Name: count, dtype: int64

In [14]:
### cell 8 ###

# Use vectorized boolean operations and masking to leverage GPU
mask = (
    (df['Math_PassStatus'] == 'F') |
    (df['Reading_PassStatus'] == 'F') |
    (df['Writing_PassStatus'] == 'F')
)

# Initialize all as 'P' then set 'F' where any fail occurs
df['OverAll_PassStatus'] = 'P'
df.loc[mask, 'OverAll_PassStatus'] = 'F'

# Compute the counts on GPU
df.OverAll_PassStatus.value_counts()

OverAll_PassStatus
P    949000
F     51000
Name: count, dtype: int64

In [15]:
### cell 9 ###

# Compute total marks via element-wise addition
cols = ['math score', 'reading score', 'writing score']
df['Total_Marks'] = df['math score'] + df['reading score'] + df['writing score']
# Compute percentage
df['Percentage'] = df['Total_Marks'] / 3

In [16]:
### cell 10 ###

# Vectorized assignment for GPU execution
df['Grade'] = 'F'
mask = df['OverAll_PassStatus'] != 'F'
df.loc[mask & (df['Percentage'] >= 40), 'Grade'] = 'E'
df.loc[mask & (df['Percentage'] >= 50), 'Grade'] = 'D'
df.loc[mask & (df['Percentage'] >= 60), 'Grade'] = 'C'
df.loc[mask & (df['Percentage'] >= 70), 'Grade'] = 'B'
df.loc[mask & (df['Percentage'] >= 80), 'Grade'] = 'A'

df.Grade.value_counts()

Grade
B    261000
C    256000
A    198000
D    178000
E     56000
F     51000
Name: count, dtype: int64

In [17]:
end_time = time.time()
print(end_time - start_time)

5.36860203742981
